# GeoJam 2026 — What Makes London Green?

![St James's Park, London](https://upload.wikimedia.org/wikipedia/commons/e/e4/St_James%27s_Park_Lake_%E2%80%93_East_from_the_Blue_Bridge_-_2012-10-06.jpg)

*St James's Park. Not every London neighbourhood looks like this.*

## The challenge

London is one of the greenest major cities in the world — roughly 47% of Greater
London is classified as green space. But that greenery is unevenly distributed.
Some neighbourhoods are lush with parks, gardens, and tree-lined streets. Others
are almost entirely concrete. And the pressure is growing: London's population has
increased by over two million since 2000, while funding for parks has declined by
more than 30%. A 2011 study estimated that London lost 3,000 hectares of garden
greenery over a decade — roughly 2.5 Hyde Parks per year.

Your team's job today: **use satellite imagery and urban data to understand what
makes a London neighbourhood green — and find the places where reality doesn't
match expectations.**

You have real Sentinel-2 satellite imagery for London from two summer dates, five
years apart. You have census and geographic data for every MSOA (Middle layer Super
Output Area — a neighbourhood-scale census unit of roughly 5,000–15,000 people).
Your task is to combine these into a machine learning model, then interrogate what
it tells you together.

### What is NDVI?

The Normalised Difference Vegetation Index is the standard remote sensing measure
of greenness. It exploits a simple fact about plants: healthy vegetation strongly
reflects near-infrared light while absorbing red light. The formula is:

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$

Values range from -1 to 1. Dense, healthy vegetation scores 0.4–0.8. Built-up
areas and bare ground sit below 0.2. Water is typically negative.

You'll compute NDVI from the raw satellite bands, aggregate it per neighbourhood,
and use it as the **target variable** for your model. The question your team
needs to answer:
*given only a neighbourhood's urban characteristics — population density, centrality,
proximity to parks — how green should it be?*

The neighbourhoods where predictions don't match reality are the interesting ones.
Surprisingly green? Maybe there's a success story — a rewilding project, a
well-maintained park. Surprisingly grey? That's a candidate for intervention.

### How to approach this

The notebook is structured in three tiers. Your team doesn't have to do them
in order, but each one builds on the previous.

- 🟢 **Beginner:** Compute NDVI from satellite bands, aggregate it to
  neighbourhoods, and map it. Get a feel for the data together.
- 🟡 **Intermediate:** Engineer features, train a model
  to predict greenness, and look at what matters.
- 🔴 **Advanced:** Use the model's errors to find anomalies — neighbourhoods
  that are unexpectedly green or grey. Build an interactive map of your findings.

**You have 2 hours. Good luck.**

In [ ]:
!pip install -q geopandas rasterstats folium mapclassify scikit-learn

## Download and load data

The data pack contains Sentinel-2 satellite bands, pre-computed NDVI rasters,
and MSOA boundaries with census-derived features. After running the next two
cells, everything is in memory — no more network calls needed.

In [ ]:
import gdown, zipfile, os, json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ========== DOWNLOAD DATA ==========
# Replace this FILE_ID with the actual Google Drive file ID
FILE_ID = "1N_8L_hBxg1u7eiif3NwFO54LINqGTXbx"

ZIP_NAME = "geojam_data.zip"
DATA_DIR = "geojam_data"

if not os.path.exists(ZIP_NAME):
    gdown.download(id=FILE_ID, output=ZIP_NAME, quiet=False)

with zipfile.ZipFile(ZIP_NAME) as zf:
    zf.extractall(".")

print("Data downloaded and extracted.")

In [ ]:
from rasterio.transform import Affine

# Raster metadata
with open(os.path.join(DATA_DIR, "raster_meta.json")) as f:
    meta = json.load(f)

raster_crs = meta["crs"]
raster_transform = Affine(*meta["transform"])

# Satellite bands (float32, reflectance 0–1)
bands_early = {b: np.load(os.path.join(DATA_DIR, f"band_early_{b}.npy")) for b in meta["band_order"]}
bands_late  = {b: np.load(os.path.join(DATA_DIR, f"band_late_{b}.npy"))  for b in meta["band_order"]}

# Pre-computed NDVI (for reference / quick look)
ndvi_early = np.load(os.path.join(DATA_DIR, "ndvi_early.npy"))
ndvi_late  = np.load(os.path.join(DATA_DIR, "ndvi_late.npy"))

# MSOA boundaries with non-satellite features
msoa = gpd.read_file(os.path.join(DATA_DIR, "msoa_base.gpkg"))

print(f"Raster shape: {meta['shape']} at {meta['resolution_m']}m resolution")
print(f"Raster CRS: {raster_crs}")
print(f"Scenes: {meta['early_date']} and {meta['late_date']}")
print(f"\nMSOA GeoDataFrame: {len(msoa)} rows")
print(f"Columns: {list(msoa.columns)}")
print(f"MSOA CRS: {msoa.crs}")

### Your data at a glance

**Raster arrays** (numpy, in EPSG:27700 — British National Grid):

| Array | Description |
|-------|-------------|
| `bands_early['B02']` .. `['B08']` | Sentinel-2 bands for the earlier year (Blue, Green, Red, NIR) |
| `bands_late['B02']` .. `['B08']` | Same bands for the later year |
| `ndvi_early`, `ndvi_late` | Pre-computed NDVI for reference |

**MSOA GeoDataFrame** (`msoa`, in EPSG:4326 — WGS84):

| Column | Description |
|--------|-------------|
| `MSOA21CD` | ONS code (unique ID) |
| `MSOA21NM` | Name, e.g. "Hackney 001" |
| `population` | Census 2021 total population |
| `area_km2` | MSOA area in km² |
| `pop_density` | People per km² |
| `dist_to_centre_km` | Distance from centroid to Charing Cross |
| `dist_to_park_km` | Distance from centroid to nearest OSM park |

**Raster metadata** (`meta`, `raster_crs`, `raster_transform`):

You'll need these when doing zonal statistics — they tell `rasterstats` how
to map pixel coordinates to geographic coordinates.

## Quick look

Just run these cells to orient yourself. No tasks here.

In [ ]:
# True colour composites (RGB = B04, B03, B02)
def make_rgb(bands, gain=3.0):
    rgb = np.stack([bands["B04"], bands["B03"], bands["B02"]], axis=-1)
    return np.clip(rgb * gain, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(make_rgb(bands_early))
axes[0].set_title(f"True colour — {meta['early_date']}")
axes[0].axis("off")
axes[1].imshow(make_rgb(bands_late))
axes[1].set_title(f"True colour — {meta['late_date']}")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# NDVI for the later year — this is what you'll be modelling
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

im = axes[0].imshow(ndvi_late, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[0].set_title(f"NDVI — {meta['late_date']}")
axes[0].axis("off")
plt.colorbar(im, ax=axes[0], fraction=0.046)

msoa.plot(ax=axes[1], edgecolor="grey", facecolor="none", linewidth=0.3)
axes[1].set_title(f"MSOA boundaries ({len(msoa)})")
axes[1].axis("off")

plt.tight_layout()
plt.show()

---
# The Challenge Starts Here
---

## 🟢 Compute NDVI from satellite bands

NDVI uses two bands: Red (`B04`) and Near-Infrared (`B08`).
The formula is `(NIR - Red) / (NIR + Red)`. Watch out for division
by zero where both bands are zero (e.g. outside the image extent).

Compute NDVI for the **later year** using `bands_late`. You can check
your result against the pre-computed `ndvi_late` array.

In [ ]:
# Your code here


## 🟢 Zonal statistics: from pixels to neighbourhoods

You now have an NDVI value for every pixel. But the model works at the
**MSOA level** — one row per neighbourhood. You need to aggregate pixel
values into each MSOA polygon.

This is the core spatial operation. Use `rasterstats.zonal_stats` to
compute the **mean NDVI** within each MSOA.

**Watch out for CRS.** The raster is in `EPSG:27700` (British National Grid)
but the MSOA GeoDataFrame is in `EPSG:4326` (WGS84). You need to reproject
the MSOAs to match the raster before computing zonal stats.

**Hints:**
- `msoa.to_crs(raster_crs)` reprojects the geometries
- `zonal_stats(geometries, raster_array, affine=raster_transform, stats=['mean'], nodata=0.0)`
- `raster_transform` and `raster_crs` are already loaded from the metadata
- Add the result as a new column `mean_ndvi` on your `msoa` GeoDataFrame
- Drop any MSOAs where `mean_ndvi` is NaN (no raster coverage)

In [ ]:
# Your code here


### 🔧 Stuck on zonal stats?

If you're hitting CRS issues or rasterstats errors, uncomment and run the
cell below to load a pre-computed version with NDVI already aggregated.
This lets you skip ahead to the mapping and modelling.

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  ESCAPE HATCH — only run if stuck on the above  ║
# ╚══════════════════════════════════════════════════╝

# msoa = gpd.read_file(os.path.join(DATA_DIR, "msoa_full.gpkg"))
# msoa = msoa.rename(columns={"mean_ndvi_late": "mean_ndvi"})
# msoa = msoa.dropna(subset=["mean_ndvi"]).reset_index(drop=True)
# print(f"Loaded pre-computed data: {len(msoa)} MSOAs")
# print(f"Columns: {list(msoa.columns)}")

## 🟢 Map greenness across London

Now that you have `mean_ndvi` per MSOA, make a choropleth map.
Which parts of London are greenest? Which are greyest?

Then find the 10 greenest and 10 greyest MSOAs. Do the results
make geographic sense?

In [ ]:
# Your code here


## 🟡 Feature engineering

You're about to build a model that predicts NDVI from **non-satellite features
only** — things like population density, distance to centre, distance to parks.
The question is: *given what we know about a neighbourhood's urban
characteristics, how green should it be?*

The base features are already in the GeoDataFrame: `pop_density`,
`dist_to_centre_km`, `dist_to_park_km`, `area_km2`. But raw features
often aren't the best inputs for a model. Transformations and combinations
can help.

**This is a good task to split across your team.** Each person engineers
one or two features, then you merge them.

Ideas:
- `log_pop_density` — density is heavily right-skewed, a log transform helps
- `dist_centre_sq` — the relationship between distance and greenness may be nonlinear
- `density_x_dist` — interaction: dense + central behaves differently from dense + suburban
- `park_access` — `dist_to_park_km / np.sqrt(area_km2)` — park distance normalised by MSOA size
- `borough` — extract the borough name from `MSOA21NM` and compute borough-level averages
- `is_inner` — binary: distance to centre < 8 km
- Anything else you think of

In [ ]:
# Your code here


## 🟡 Predict greenness with ML

Train a model to predict `mean_ndvi` from **non-satellite features only**.
The question: *given what we know about a neighbourhood's urban characteristics,
how green should it be?*

Look at feature importance once you have a model — what drives greenness in London?

In [ ]:
# Your code here


## 🔴 Find the anomalies

Your model predicts how green each neighbourhood *should* be. Some will
be greener than predicted, others greyer. Those gaps — the residuals —
are where the interesting stories are.

Map them. Investigate the extremes. What's going on in the places where
the model gets it most wrong?

In [ ]:
# Your code here


## Final deliverable

Build an interactive map of your anomaly scores using Folium. The map
should let someone hover over an MSOA and see its name, actual NDVI,
predicted NDVI, and residual.

Then prepare a short write-up as a team:
- Your **top 5 "surprisingly grey" MSOAs** — where should London intervene?
- One **"surprisingly green" success story** — what might explain it?
- One **caveat** about your model or data

In [ ]:
# Your code here


## Going further

If you've finished the main challenge, here are some directions to explore:

- **Add ΔNDVI:** you have both years' imagery. Compute the change in NDVI
  between 2019 and 2024 and model that too. Where is London getting greener
  or greyer over time?
- **Sub-MSOA analysis:** work at pixel level instead of MSOA level. Cluster
  pixels by their spectral profile. How does land cover composition vary
  across neighbourhoods?
- **Spectral features:** derive features from the satellite bands themselves.
  What proportion of each MSOA is water? Bare soil? What does NDVI
  heterogeneity (standard deviation) tell you?
- **Multi-temporal:** the data pack has only two scenes. With more scenes
  across years you could build a proper time series of greenness change.
- **Equity lens:** cross your greenness map with population density. Which
  communities have the least access to green space per person?